## 1. Mount Drive

In [1]:
import sys
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
try:
    from PIL import Image, ImageOps
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'Pillow>=10'])
print('Drive mounted. Complete')

Mounted at /content/drive
Drive mounted. Complete


## 2. Edit your configuration


In [2]:
PROJECT_DIR = Path('/content/drive/MyDrive/Frugal AI Project')
TEAM_DIR = PROJECT_DIR / 'Human-review/human_review/team_results'

REVIEWER_SLOT = 'reviewer_1' #here
REVIEWER_NAME = 'Salma' #here
ADD_GLASSES = True

if not (PROJECT_DIR / 'sampled_dataset.csv').is_file():
    raise FileNotFoundError('Check PROJECT_DIR: sampled_dataset.csv was not found.')
for name in ['assignments.csv', 'baseline_reviews.csv']:
    if not (TEAM_DIR / name).is_file():
        raise FileNotFoundError(f'Missing {TEAM_DIR / name}. Correct TEAM_DIR; do not recreate your results.')
if REVIEWER_SLOT not in ['reviewer_1', 'reviewer_2', 'reviewer_3']:
    raise ValueError('Enter your agreed REVIEWER_SLOT in this cell.')
if not REVIEWER_NAME.strip():
    raise ValueError('Enter your REVIEWER_NAME in this cell.')
print('Reviewer:', REVIEWER_NAME, '| Slot:', REVIEWER_SLOT)
print('Results:', TEAM_DIR / f'{REVIEWER_SLOT}.csv')

Reviewer: Salma | Slot: reviewer_1
Results: /content/drive/MyDrive/Frugal AI Project/Human-review/human_review/team_results/reviewer_1.csv


## 3. Load the complete review engine

In [3]:
import csv
import hashlib
import html
import io
import json
import os
from pathlib import Path
from datetime import datetime, timezone

REVIEW_FIELDS = ['HASH', 'accessory', 'reviewer', 'correct_accessory', 'placement_ok',
                'portrait_preserved', 'visual_defects', 'decision', 'notes',
                'ETHNICITY', 'GENDER', 'AGE', 'original_path', 'generated_path',
                'reviewed_at', 'reviewer_slot']
CHECKS = ['correct_accessory', 'placement_ok', 'portrait_preserved', 'visual_defects']


def read_rows(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


def key(row):
    return row['HASH'], row['accessory']


def fingerprint(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest() if Path(path).exists() else None


def write_csv(path, rows, fields=REVIEW_FIELDS):
    path = Path(path)
    temporary = path.with_suffix('.csv.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)
        f.flush()
        os.fsync(f.fileno())
    os.replace(temporary, path)


def index_glasses(project):
    folder = Path(project)/'generated_glasses'
    indexed = {}
    if folder.is_dir():
        for p in folder.iterdir():
            if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg'}:
                indexed.setdefault(p.stem.removesuffix('_glasses'), []).append(p)
    return indexed


def glasses_path(project, h, indexed=None):
    folder = Path(project)/'generated_glasses'
    matches = (index_glasses(project) if indexed is None else indexed).get(h, [])
    if len(matches)!=1:
        raise ValueError(f'Expected one glasses image for {h} in {folder}; found {len(matches)}. Supported names: <HASH>_glasses.png or <HASH>.jpg (PNG/JPG/JPEG).')
    return matches[0]


def add_glasses_assignments(project, team_dir, apply=False):
    project, team_dir = Path(project),Path(team_dir)
    path=team_dir/'assignments.csv'
    before=path.read_bytes()
    assignments=read_rows(path)
    owners={key(r):r['reviewer_slot'] for r in assignments}
    if len(owners)!=len(assignments):raise ValueError('Duplicate existing assignments.')
    sample=read_rows(project/'sampled_dataset.csv')
    ids={r['HASH'] for r in sample}
    if len(ids)!=len(sample):raise ValueError('Duplicate sample IDs.')
    if {k for k in owners if k[1]!='glasses'} != {(h,a) for h in ids for a in ['mask','cap']}:
        raise ValueError('Existing assignments do not match the mask/cap sample.')
    if any(h not in ids for h,a in owners):raise ValueError('Unexpected assigned ID.')
    slots=sorted({r['reviewer_slot'] for r in assignments})
    # Validate every requested file before changing anything.
    indexed = index_glasses(project)
    for h in sorted(ids):glasses_path(project,h,indexed)
    additions=[]
    groups={}
    for row in sample:groups.setdefault(tuple(row[k] for k in ['ETHNICITY','GENDER','AGE']),[]).append(row)
    for group in sorted(groups):
        for i,row in enumerate(sorted(groups[group],key=lambda r:r['HASH'])):
            if (row['HASH'],'glasses') not in owners:
                additions.append({'HASH':row['HASH'],'accessory':'glasses','reviewer_slot':slots[(i+2)%len(slots)]})
    if apply and additions:
        if path.read_bytes()!=before:raise ValueError('Assignments changed. Stop and retry after all sessions are closed.')
        backup=team_dir/('assignments_before_glasses_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.csv.bak')
        backup.write_bytes(before)
        write_csv(path,assignments+additions,['HASH','accessory','reviewer_slot'])
    return {'new_glasses_assignments':len(additions),'total_after_update':len(assignments)+len(additions),
            'applied':bool(apply),'existing_reviews_modified':False}


class TeamReview:
    def __init__(self, project, team_dir, slot, reviewer):
        self.project, self.team_dir = Path(project), Path(team_dir)
        self.slot, self.reviewer = slot.strip(), reviewer.strip()
        if not self.reviewer or len(self.reviewer) > 80:
            raise ValueError('Set REVIEWER_NAME to your own name or ID (1–80 characters).')
        self.assignments_path = self.team_dir / 'assignments.csv'
        self.assignment_digest = fingerprint(self.assignments_path)
        self.assignments = read_rows(self.assignments_path)
        slots = {r['reviewer_slot'] for r in self.assignments}
        if self.slot not in slots:
            raise ValueError(f'Choose one of the existing slots: {sorted(slots)}')
        if any(not s.replace('_','').isalnum() for s in slots):
            raise ValueError('Invalid reviewer slot in assignment file.')
        self.slots = slots
        self.owners = {key(r):r['reviewer_slot'] for r in self.assignments}
        if len(self.owners) != len(self.assignments):
            raise ValueError('Duplicate assignments. Do not edit assignments.csv.')
        metadata = read_rows(self.project / 'sampled_dataset.csv')
        self.metadata = {r['HASH']:r for r in metadata}
        if len(metadata) != len(self.metadata):
            raise ValueError('Duplicate sample IDs.')
        self.accessories = sorted({a for h,a in self.owners})
        if not {'mask','cap'}<=set(self.accessories)<={'mask','cap','glasses'}:raise ValueError('Unknown or missing accessory types in assignments.')
        expected = {(h,a) for h in self.metadata for a in self.accessories}
        if set(self.owners) != expected:
            raise ValueError('Assignments do not match this sample. Use the original sample and assignment file.')
        self.glasses_index = index_glasses(self.project) if 'glasses' in self.accessories else {}
        self.path = self.team_dir / f'{self.slot}.csv'
        self.disk_digest = fingerprint(self.path)
        self.own = {key(r):r for r in read_rows(self.path)} if self.path.exists() else {}
        if self.own and {r['reviewer'] for r in self.own.values()} != {self.reviewer}:
            raise ValueError('This slot already belongs to a different reviewer name. Use your assigned slot and the same name when resuming.')
        self.all_reviews()  # Validate all existing files before displaying any work.

    def all_reviews(self):
        if fingerprint(self.assignments_path) != self.assignment_digest:
            raise ValueError('Assignments changed during this session. Stop this interface and rerun the assignment-check and interface cells to load the updated assignments.')
        completed = {}
        baseline = self.team_dir / 'baseline_reviews.csv'
        if not baseline.is_file():
            raise FileNotFoundError('Missing baseline_reviews.csv. Restore the complete team_results folder.')
        for path in [baseline] + [self.team_dir / f'{slot}.csv' for slot in sorted(self.slots)]:
            if not path.exists():
                continue
            for row in read_rows(path):
                k = key(row)
                if k not in self.owners:
                    raise ValueError(f'Unexpected image in {path.name}: {k}')
                if row.get('decision') not in ['accept','reject','uncertain']:
                    raise ValueError(f'Invalid saved decision in {path.name}')
                if path != baseline and (self.owners[k] != path.stem or row.get('reviewer_slot') != path.stem):
                    raise ValueError(f'Review outside its assigned slot: {path.name}, {k}')
                if k in completed:
                    raise ValueError(f'Duplicate completed image found: {k}. Ask the coordinator to reconcile; no rows were changed.')
                completed[k] = row
        return completed

    def queue(self, accessory='all', saved=False):
        completed = self.all_reviews()
        return [key(r) for r in self.assignments if r['reviewer_slot']==self.slot
                and (accessory=='all' or r['accessory']==accessory)
                and (key(r) in self.own if saved else key(r) not in completed)]

    def paths(self, item):
        h, accessory = item
        original = self.project / 'generated_portraits_hashed_final/generated_portraits_hashed' / f'{h}.jpg'
        if accessory=='glasses':
            generated=glasses_path(self.project,h,self.glasses_index)
        else:
            folder, suffix = ('generated_masks','mask') if accessory=='mask' else ('generated_cap','hat')
            generated = self.project / folder / f'{h}_{suffix}.png'
        if not original.is_file() or not generated.is_file():
            raise FileNotFoundError(f'Missing original or generated image for {h} / {accessory}')
        return original, generated

    def save(self, item, answers, decision, notes):
        if self.owners.get(item) != self.slot:
            raise ValueError('This image is not assigned to your slot.')
        if fingerprint(self.path) != self.disk_digest:
            raise ValueError('Your results file changed in another session. Stop here, close other sessions for this slot, and rerun the notebook before saving.')
        completed = self.all_reviews()
        if item in completed and item not in self.own:
            raise ValueError('This image is already reviewed; refresh the queue.')
        if any(answers.get(k) not in ['yes','no','uncertain'] for k in CHECKS):
            raise ValueError('Answer all four checks.')
        if decision not in ['accept','reject','uncertain']:
            raise ValueError('Choose Accept, Reject, or Uncertain.')
        if decision=='accept' and [answers[k] for k in CHECKS] != ['yes','yes','yes','no']:
            raise ValueError('Accept requires Yes on the first three checks and No visual defects.')
        notes = notes.strip()
        if decision!='accept' and not notes:
            raise ValueError('Add a short reason for Reject or Uncertain.')
        if len(notes)>4000:
            raise ValueError('Limit notes to 4,000 characters.')
        original, generated = self.paths(item)
        row = {**self.metadata[item[0]], **answers, 'accessory':item[1], 'reviewer':self.reviewer,
               'reviewer_slot':self.slot, 'decision':decision, 'notes':notes,
               'original_path':str(original), 'generated_path':str(generated),
               'reviewed_at':datetime.now(timezone.utc).isoformat(timespec='seconds')}
        # Only this slot's file is ever written. Other reviewers own different files.
        next_rows = dict(self.own)
        next_rows[item] = row
        if self.path.exists():
            backup = self.path.with_suffix('.csv.bak')
            backup.write_bytes(self.path.read_bytes())
        write_csv(self.path, next_rows.values())
        self.own = next_rows
        self.disk_digest = fingerprint(self.path)
        reread = {key(r):r for r in read_rows(self.path)}
        if reread.get(item) != {f:str(row.get(f,'')) for f in REVIEW_FIELDS}:
            raise OSError('Save verification failed. Check the Drive results file before continuing.')
        return row

    def progress(self):
        completed = self.all_reviews()
        assigned = {k for k,owner in self.owners.items() if owner==self.slot}
        return f'Team: {len(completed)} / {len(self.owners)} completed. Your slot: {len(assigned & set(completed))} / {len(assigned)} completed.'


print('Review engine loaded: mask, cap, glasses.')

Review engine loaded: mask, cap, glasses.


## 4. Validate assignments and resume progress

In [6]:
from collections import Counter

if ADD_GLASSES:
    print(add_glasses_assignments(PROJECT_DIR, TEAM_DIR, apply=True))

assignment_counts = Counter(r['accessory'] for r in read_rows(TEAM_DIR / 'assignments.csv'))
print('Saved assignments:', dict(assignment_counts))
sample_count = len(read_rows(PROJECT_DIR / 'sampled_dataset.csv'))
session = TeamReview(PROJECT_DIR, TEAM_DIR, REVIEWER_SLOT, REVIEWER_NAME)
print(session.progress())
for accessory in session.accessories:
    print(f'Your remaining {accessory}: {len(session.queue(accessory))}')

{'new_glasses_assignments': 0, 'total_after_update': 2160, 'applied': True, 'existing_reviews_modified': False}
Saved assignments: {'mask': 720, 'cap': 720, 'glasses': 720}
Team: 690 / 2160 completed. Your slot: 253 / 720 completed.
Your remaining cap: 156
Your remaining glasses: 156
Your remaining mask: 155


## 5. Open the review interface

In [9]:

import base64
import io
import json
import threading
import uuid
from PIL import Image,ImageOps
from IPython.display import HTML,JSON,display
from google.colab import output

missing=[
    name for name in [
        'TeamReview','PROJECT_DIR','TEAM_DIR',
        'REVIEWER_SLOT','REVIEWER_NAME'
    ]
    if name not in globals()
]
if missing:
    raise RuntimeError(
        f'Missing: {", ".join(missing)}.'
    )

session=TeamReview(PROJECT_DIR,TEAM_DIR,REVIEWER_SLOT,REVIEWER_NAME)

old_callback=globals().get("_portrait_callback_name")
if old_callback:
    try:
        output.unregister_callback(old_callback)
    except ValueError:
        pass

callback_name="portrait_review_"+uuid.uuid4().hex
_portrait_callback_name=callback_name
callback_lock=threading.RLock()

def image_data(path):
    with Image.open(path) as image:
        image=ImageOps.exif_transpose(image).convert("RGB")
        buffer=io.BytesIO()
        image.save(buffer,format="PNG")
    return "data:image/png;base64,"+base64.b64encode(buffer.getvalue()).decode()

def review_callback(action,payload):
    with callback_lock:
        try:
            if action=="save":
                session.save(
                    tuple(payload["item"]),
                    payload["answers"],
                    payload["decision"],
                    payload["notes"],
                )
                return JSON({
                    "ok":True,
                    "message":f"Saved and verified in {session.path.name}.",
                })

            if action!="load":
                raise ValueError("Unknown action.")

            accessory=payload.get("accessory","all")
            queue=session.queue(
                accessory,
                payload.get("mode","new")=="saved",
            )
            index=min(max(int(payload.get("index",0)),0),len(queue)-1)

            result={
                "ok":True,
                "progress":session.progress(),
                "reviewer":session.reviewer,
                "path":str(session.path),
                "total":len(queue),
                "index":max(index,0),
                "item":None,
                "empty_message":(
                    "No saved reviews in this view yet. Choose Unreviewed to start."
                    if payload.get("mode","new")=="saved" else
                    "No unreviewed images remain for your slot in this view. "
                    "Choose another accessory or My saved reviews."
                ),
            }

            if queue:
                item=queue[index]
                original,generated=session.paths(item)
                result.update({
                    "item":list(item),
                    "original":image_data(original),
                    "generated":image_data(generated),
                    "review":session.own.get(item),
                })

            return JSON(result)

        except Exception as error:
            return JSON({"ok":False,"error":str(error)})

output.register_callback(callback_name,review_callback)

page=r"""
<div id="team_review">
<style>
#team_review {
  font-family:system-ui,Segoe UI,sans-serif;
  color:#20332e;background:#f4f6f2;padding:5px;
  overflow-x:auto;font-size:12px;
}
#team_review * {box-sizing:border-box}
#team_review .toolbar {
  display:flex;align-items:center;gap:5px;flex-wrap:wrap;margin-bottom:4px;
}
#team_review select, #team_review button, #team_review textarea {
  font:inherit;border:1px solid #d3ddd4;border-radius:8px;
  background:white;color:#20332e;padding:5px 7px;
}
#team_review button {cursor:pointer}
#team_review button:disabled {opacity:.45;cursor:default}
#team_review .workspace {
  display:grid;grid-template-columns:minmax(0,1fr) 285px;
  gap:8px;min-width:0;
}
#team_review .panel {
  background:white;border:1px solid #d8e1d9;border-radius:13px;padding:8px;
  min-width:0;
}
#team_review .meta {
  display:flex;justify-content:space-between;align-items:center;
  gap:8px;font-size:11px;color:#617069;margin-bottom:5px;flex-wrap:wrap;
}
#team_review .tag {
  border-radius:25px;background:#e6f0e8;color:#17664b;
  padding:4px 9px;font-weight:700;
}
#team_review .pair {display:grid;grid-template-columns:1fr 1fr;gap:8px}
#team_review figure {margin:0;min-width:0}
#team_review figcaption {
  display:flex;justify-content:space-between;align-items:center;
  font-size:12px;font-weight:650;margin-bottom:5px;gap:4px;flex-wrap:wrap;
}
#team_review .zoom {font-size:11px;font-weight:400;color:#617069}
#team_review .portrait {
  display:block;width:100%;height:220px;min-height:0;
  object-fit:contain;background:#edf0ed;border-radius:11px;cursor:zoom-in;
}
#team_review h2 {font-size:16px;margin:0 0 2px}
#team_review .muted {color:#617069;font-size:11px;line-height:1.35}
#team_review .intro {margin:0 0 6px}
#team_review fieldset {border:0;padding:0;margin:0 0 6px}
#team_review legend {font-size:11px;font-weight:700;line-height:1.25;margin-bottom:3px}
#team_review .choices {display:flex;gap:5px}
#team_review .choice {flex:1;position:relative}
#team_review .choice input {position:absolute;opacity:0}
#team_review .choice span {
  display:block;text-align:center;padding:4px 2px;
  border:1px solid #d3ddd4;border-radius:7px;
  font-size:11px;cursor:pointer;background:white;
}
#team_review .choice input:checked + span {
  background:#e3f0e7;border-color:#17664b;color:#17664b;font-weight:700;
}
#team_review textarea {
  width:100%;height:36px;min-height:36px;resize:vertical;
  margin-top:3px;padding:5px;
}
#team_review .save {
  width:100%;background:#17664b;color:white;border-color:#17664b;
  font-weight:700;margin-top:5px;padding:8px;font-size:12px;
}
#team_review .navigation {
  display:flex;align-items:center;justify-content:space-between;
  gap:6px;margin-top:6px;
}
#team_review .hint {margin-top:6px}
#team_review details {font-size:11px;line-height:1.4;margin-top:6px}
#team_review summary {cursor:pointer;font-weight:700}
#team_review .message {
  white-space:pre-wrap;font-size:11px;margin:4px 0;
  padding:4px 7px;border-radius:8px;background:#e6f0e8;color:#17664b;
}
#team_review .message:empty {display:none}
#team_review .message.error {background:#fff0e9;color:#9b3020}
#team_review .path {overflow-wrap:anywhere;margin:6px 0 0;font-size:10px}
#team_review dialog {border:0;border-radius:12px;padding:15px;max-width:95vw}
#team_review dialog img {display:block;max-width:90vw;max-height:82vh;object-fit:contain}
#team_review dialog::backdrop {background:#0009}
@media(max-width:650px) {
  #team_review .workspace {grid-template-columns:minmax(0,1fr) 255px;gap:5px}
  #team_review .portrait {height:110px}
}
</style>
<div class="toolbar">
  <strong data-id="reviewer"></strong>
  <select data-id="accessory">
    <option value="all">All accessories</option>
    <option value="mask">Masks</option>
    <option value="cap">Caps</option>
    <option value="glasses">Glasses</option>
  </select>
  <select data-id="mode">
    <option value="new">Unreviewed</option>
    <option value="saved">My saved reviews</option>
  </select>
  <button data-id="refresh">Load / refresh</button>
  <span class="muted" data-id="progress"></span>
</div>
<div class="message" data-id="message"></div>
<div data-id="empty" class="panel" hidden></div>
<div class="workspace" data-id="workspace">
  <section class="panel">
    <div class="meta">
      <span class="tag" data-id="tag"></span>
      <span data-id="hash"></span>
      <span data-id="saved"></span>
    </div>
    <div class="pair">
      <figure>
        <figcaption>Original portrait <span class="zoom">Click to enlarge ↗</span></figcaption>
        <img class="portrait" data-id="original" alt="Original portrait"
             tabindex="0" loading="eager" decoding="sync">
      </figure>
      <figure>
        <figcaption>Generated portrait <span class="zoom">Click to enlarge ↗</span></figcaption>
        <img class="portrait" data-id="generated" alt="Generated portrait"
             tabindex="0" loading="eager" decoding="sync">
      </figure>
    </div>
    <div class="navigation">
      <button data-id="previous">← Previous</button>
      <span class="muted" data-id="position"></span>
      <button data-id="next">Skip / next →</button>
    </div>
    <p class="muted hint">
      Judge visible facial features against the original. Expected covering
      by an accessory is not a defect. Click either image to inspect it.
    </p>
    <details>
      <summary>Review guide</summary>
      <p>Check that the requested accessory is present and naturally placed.
      Compare visible facial features and unrelated details with the original.
      Look for distortion, broken parts, or unnatural blending.</p>
      <p>Accept requires Yes to the first three checks and No visual defects.
      Reject clear failures. Choose Uncertain when another opinion is needed.</p>
    </details>
  </section>
  <section class="panel">
    <h2>Your assessment</h2>
    <p class="muted intro">Complete all four checks, then decide.</p>
    <form data-id="form">
      <div data-id="questions"></div>
      <fieldset>
        <legend>Overall decision</legend>
        <div class="choices" data-id="decisions"></div>
      </fieldset>
      <label><b>Notes</b>
        <div class="muted">Required for reject or uncertain</div>
        <textarea data-id="notes" maxlength="4000"
          placeholder="Describe any issue you observed…"></textarea>
      </label>
      <button class="save" data-id="save" type="submit">
        Save & next unreviewed →
      </button>
      <p class="muted">Only Save records an assessment. Skip leaves it unreviewed.</p>
    </form>
  </section>
</div>
<p class="muted path" data-id="path"></p>
<dialog data-id="dialog">
  <button data-id="close">Close ✕</button>
  <img data-id="large" alt="Enlarged portrait">
</dialog>
<script>
(async () => {
  const root = document.getElementById("team_review");
  const get = id => root.querySelector(`[data-id="${id}"]`);
  const questions = [
    ["correct_accessory", "1. Is the intended accessory present?"],
    ["placement_ok", "2. Is its placement natural?"],
    ["portrait_preserved", "3. Is the portrait otherwise preserved?"],
    ["visual_defects", "4. Are there visual defects?"]
  ];
  function choice(name, value, label) {
    return `<label class="choice"><input type="radio" name="${name}"
      value="${value}" required><span>${label}</span></label>`;
  }
  get("questions").innerHTML = questions.map(([key, title]) =>
    `<fieldset><legend>${title}</legend><div class="choices">
      ${choice(key, "yes", "Yes")}
      ${choice(key, "no", "No")}
      ${choice(key, "uncertain", "Uncertain")}
    </div></fieldset>`
  ).join("");
  get("decisions").innerHTML =
    choice("decision", "accept", "Accept") +
    choice("decision", "reject", "Reject") +
    choice("decision", "uncertain", "Uncertain");
  let current = null, index = 0, total = 0;
  let dirty = false, busy = false, ready = false;
  let activeAccessory = "all", activeMode = "new";
  function message(text, error=false) {
    get("message").textContent = text;
    get("message").classList.toggle("error", error);
  }
  function controls() {
    root.querySelectorAll("button,select,input,textarea")
      .forEach(el => el.disabled = busy);
    get("save").disabled = busy || !current || !ready;
    get("previous").disabled = busy || !current || index === 0;
    get("next").disabled = busy || !current || index >= total - 1;
  }
  async function call(action, payload) {
    const response = await google.colab.kernel.invokeFunction(
      "__CALLBACK__", [action, payload], {}
    );
    const result = response.data["application/json"];
    if (!result.ok) throw new Error(result.error);
    return result;
  }
  function mayLeave() {
    return !dirty || confirm("Discard these unsaved answers?");
  }
  async function renderImage(element, source) {
    element.src = "";
    await new Promise(requestAnimationFrame);
    await new Promise((resolve, reject) => {
      element.onload = resolve;
      element.onerror = () => reject(
        new Error("Portrait image could not load.")
      );
      element.src = source;
    });
    element.style.visibility = "hidden";
    void element.offsetWidth;
    element.style.visibility = "visible";
    await new Promise(requestAnimationFrame);
    await new Promise(requestAnimationFrame);
  }
  async function load(at) {
    const data = await call("load", {
      accessory: activeAccessory,
      mode: activeMode,
      index: at
    });
    current = data.item;
    index = data.index;
    total = data.total;
    dirty = false;
    ready = false;
    get("form").reset();
    get("reviewer").textContent = data.reviewer;
    get("progress").textContent = data.progress;
    get("path").textContent = "Results saved to: " + data.path;
    get("workspace").hidden = !current;
    get("workspace").style.display = current ? "grid" : "none";
    get("empty").hidden = Boolean(current);
    get("empty").textContent = data.empty_message;
    if (!current) return;
    get("tag").textContent =
      current[1] === "cap" ? "BASEBALL CAP" : current[1].toUpperCase();
    get("hash").textContent = current[0];
    get("saved").textContent =
      data.review ? "Saved: " + data.review.decision : "Not reviewed";
    get("position").textContent = `${index + 1} of ${total} in this view`;
    if (data.review) {
      for (const key of [...questions.map(q => q[0]), "decision"]) {
        const input = root.querySelector(
          `input[name="${key}"][value="${data.review[key]}"]`
        );
        if (input) input.checked = true;
      }
      get("notes").value = data.review.notes || "";
    }
    await Promise.all([
      renderImage(get("original"), data.original),
      renderImage(get("generated"), data.generated)
    ]);
    ready = true;
  }
  async function navigate(at, changeFilters=false) {
    if (busy || !mayLeave()) return;
    busy = true;
    controls();
    try {
      if (changeFilters) {
        activeAccessory = get("accessory").value;
        activeMode = get("mode").value;
      }
      await load(at);
      message("Ready. Complete the checks, then save.");
    } catch (error) {
      ready = false;
      message("Could not load: " + error.message, true);
    } finally {
      busy = false;
      controls();
    }
  }
  get("form").addEventListener("input", () => dirty = true);
  get("refresh").onclick = () => navigate(0, true);
  get("previous").onclick = () => navigate(index - 1);
  get("next").onclick = () => navigate(index + 1);
  get("form").onsubmit = async event => {
    event.preventDefault();
    if (busy || !current || !ready) return;
    const values = Object.fromEntries(new FormData(get("form")));
    const answers = Object.fromEntries(
      questions.map(([key]) => [key, values[key]])
    );
    busy = true;
    controls();
    let saved = false;
    try {
      const result = await call("save", {
        item: current,
        answers,
        decision: values.decision,
        notes: get("notes").value
      });
      saved = true;
      dirty = false;
      message(result.message);
      await load(activeMode === "saved" ? index + 1 : index);
      message(result.message);
    } catch (error) {
      if (saved) {
        ready = false;
        message(
          "Your review was saved, but the next image could not load. " +
          "Click Load / refresh. " + error.message,
          true
        );
      } else {
        message("Not saved: " + error.message, true);
      }
    } finally {
      busy = false;
      controls();
    }
  };
  for (const id of ["original", "generated"]) {
    const open = () => {
      get("large").src = get(id).src;
      get("dialog").showModal();
    };
    get(id).onclick = open;
    get(id).onkeydown = event => {
      if (event.key === "Enter") open();
    };
  }
  get("close").onclick = () => get("dialog").close();
  window.addEventListener("beforeunload", event => {
    if (dirty || busy) {
      event.preventDefault();
      event.returnValue = "";
    }
  });
  await navigate(0);
})();
</script>
</div>
"""

page=page.replace("__CALLBACK__",callback_name)
display(HTML(page))

## 6. Combine all saved reviews into one CSV

In [10]:
from pathlib import Path
import csv
import os
from datetime import datetime, timezone

def read_review_csv(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

assignment_rows = read_review_csv(TEAM_DIR / 'assignments.csv')
assigned = {(row['HASH'], row['accessory']) for row in assignment_rows}
slots = sorted({row['reviewer_slot'] for row in assignment_rows})
input_files = [TEAM_DIR / 'baseline_reviews.csv'] + [TEAM_DIR / f'{slot}.csv' for slot in slots]

combined_rows = []
seen = {}
for path in input_files:
    if not path.exists():
        continue
    for row in read_review_csv(path):
        item = (row.get('HASH', ''), row.get('accessory', ''))
        if item not in assigned:
            raise ValueError(f'Unexpected review in {path.name}: {item}. Nothing was written.')
        if row.get('decision') not in {'accept', 'reject', 'uncertain'}:
            raise ValueError(f'Invalid decision in {path.name}: {item}. Nothing was written.')
        if item in seen:
            raise ValueError(
                f'Duplicate decision for {item} in {seen[item]} and {path.name}. '
                'Resolve this before combining; nothing was written.'
            )
        seen[item] = path.name
        combined_rows.append(row)

columns = REVIEW_FIELDS
destination = TEAM_DIR / 'combined_human_review.csv'
temporary = TEAM_DIR / 'combined_human_review.csv.tmp'
with temporary.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(sorted(combined_rows, key=lambda r: (r['HASH'], r['accessory'])))
    handle.flush()
    os.fsync(handle.fileno())
os.replace(temporary, destination)

print('Combined review snapshot:', destination)
print(f'{len(combined_rows)} saved reviews / {len(assigned)} assigned images')
print('Created UTC:', datetime.now(timezone.utc).isoformat(timespec='seconds'))

Combined review snapshot: /content/drive/MyDrive/Frugal AI Project/Human-review/human_review/team_results/combined_human_review.csv
690 saved reviews / 2160 assigned images
Created UTC: 2026-10-12T22:01:16+00:00
